# Exploitation Zone: Building the Knowledge Graph

Falta:
- Millorar integració dels diferents data sources
- Els ratings que estan parsejats per separat ara mateix
- Fer Oscar winner / Oscar nominated
- Pensar més possibles relacions
- Definir domain i range de la majoria
- Relació artist - crew member (el compositor?)
- Crew member i actors tenen més info diria

1.  **Setup**
2.  **Schema Definition (Ontology)**
3.  **Graph Population (Triplification)**
4.  **Serialization**

## 1. Setup and Initialization

In [96]:
import pandas as pd
from sqlalchemy import create_engine
from rdflib import Graph, Namespace, URIRef, Literal, XSD
from rdflib.namespace import RDF, RDFS, FOAF
import re
from urllib.parse import quote_plus

In [97]:
db_user = "postgres"
db_password = "postgres"
db_host = "localhost"
db_port = "5432"
db_name = "bdaproject2"

db_connection_str = f"postgresql+psycopg2://{db_user}:{quote_plus(db_password)}@{db_host}:{db_port}/{db_name}"
db_engine = create_engine(db_connection_str)

print("Loading trusted data from PostgreSQL...")
try:
    movies_df = pd.read_sql("SELECT * FROM movies", db_engine)
    movie_genres_df = pd.read_sql("SELECT * FROM movie_genres", db_engine)
    movie_companies_df = pd.read_sql("SELECT * FROM movie_production_companies", db_engine)
    movie_cast_df = pd.read_sql("SELECT * FROM movie_cast", db_engine)
    movie_crew_df = pd.read_sql("SELECT * FROM movie_crew", db_engine)
    omdb_details_df = pd.read_sql("SELECT * FROM omdb_movie_details", db_engine)
    actor_articles_df = pd.read_sql("SELECT * FROM actor_articles", db_engine)
    article_keywords_df = pd.read_sql("SELECT * FROM article_keyword", db_engine)
    music_tracks_df = pd.read_sql("SELECT * FROM music_tracks", db_engine)
    track_genres_df = pd.read_sql("SELECT * FROM track_genre", db_engine)
    print("Data loaded successfully!")
    print(f"Movies loaded: {len(movies_df)}")
except Exception as e:
    print(f"An error occurred while loading data: {e}")

Loading trusted data from PostgreSQL...
Data loaded successfully!
Movies loaded: 1344


## 2. Schema Definition
`FOAF` ho ha dit el chat

In [ ]:
g = Graph()

BDA = Namespace("http://www.bdaproject.org/schema#")
INST = Namespace("http://www.bdaproject.org/instance/")
SCHEMA = Namespace("http://schema.org/")
DBO = Namespace("http://dbpedia.org/ontology/")

g.bind("bda", BDA)
g.bind("inst", INST)
g.bind("schema", SCHEMA)
g.bind("foaf", FOAF)
g.bind("dbo", DBO)

g.add((BDA.Movie, RDF.type, RDFS.Class))
g.add((BDA.Person, RDF.type, RDFS.Class))
g.add((BDA.Actor, RDFS.subClassOf, BDA.Person))
g.add((BDA.CrewMember, RDFS.subClassOf, BDA.Person))
g.add((BDA.Genre, RDF.type, RDFS.Class))
g.add((BDA.ProductionCompany, RDF.type, RDFS.Class))
g.add((BDA.MovieCollection, RDF.type, RDFS.Class))
g.add((BDA.Article, RDF.type, RDFS.Class))
g.add((BDA.SoundtrackAlbum, RDF.type, RDFS.Class))
g.add((BDA.SoundtrackTrack, RDF.type, RDFS.Class))
g.add((BDA.Artist, RDF.type, RDFS.Class))

props = {
    "hasActor": (BDA.Movie, BDA.Actor),
    "hasCrewMember": (BDA.Movie, BDA.CrewMember),
    "hasGenre": (BDA.Movie, BDA.Genre),
    "producedBy": (BDA.Movie, BDA.ProductionCompany),
    "belongsToCollection": (BDA.Movie, BDA.MovieCollection),
    "isDiscussedIn": (BDA.Person, BDA.Article),
    "hasSoundtrackAlbum": (BDA.Movie, BDA.SoundtrackAlbum),
    "hasTrack": (BDA.SoundtrackAlbum, BDA.SoundtrackTrack),
    "performedBy": (BDA.SoundtrackTrack, BDA.Artist)
}
for p, (domain, range) in props.items():
    prop_uri = BDA[p]
    g.add((prop_uri, RDF.type, RDF.Property))
    g.add((prop_uri, RDFS.domain, domain))
    g.add((prop_uri, RDFS.range, range))

g.add((SCHEMA.name, RDF.type, RDF.Property))
g.add((SCHEMA.description, RDF.type, RDF.Property))
g.add((SCHEMA.datePublished, RDF.type, RDF.Property))
g.add((DBO.budget, RDF.type, RDF.Property))
g.add((DBO.revenue, RDF.type, RDF.Property))
g.add((SCHEMA.duration, RDF.type, RDF.Property))
g.add((SCHEMA.ratingValue, RDF.type, RDF.Property))
g.add((SCHEMA.ratingCount, RDF.type, RDF.Property))
g.add((SCHEMA.boxOffice, RDF.type, RDF.Property))
g.add((BDA.metascore, RDF.type, RDF.Property))
g.add((BDA.awardWins, RDF.type, RDF.Property))
g.add((BDA.awardNominations, RDF.type, RDF.Property))

print("Schema and ontology defined.")

Schema and ontology defined.


## 3. Graph Population (Triplification)

In [99]:
def add_literal(graph, subject, predicate, value, datatype):
    if value is not None and pd.notna(value):
        graph.add((subject, predicate, Literal(value, datatype=datatype)))
        
print("Populating graph from 'movies' table...")
for _, row in movies_df.iterrows():
    movie_uri = INST[f'movie/{row["movie_id"]}']
    g.add((movie_uri, RDF.type, BDA.Movie))

    add_literal(g, movie_uri, SCHEMA.name, row['title'].replace('"',''), XSD.string)
    add_literal(g, movie_uri, SCHEMA.description, row['overview'], XSD.string)
    add_literal(g, movie_uri, SCHEMA.datePublished, row['release_date'], XSD.date)
    add_literal(g, movie_uri, DBO.budget, row['budget'], XSD.long)
    add_literal(g, movie_uri, DBO.revenue, row['revenue'], XSD.long)
    add_literal(g, movie_uri, SCHEMA.duration, row['runtime'], XSD.integer)
    add_literal(g, movie_uri, SCHEMA.ratingValue, row['vote_average'], XSD.float)
    add_literal(g, movie_uri, SCHEMA.ratingCount, row['vote_count'], XSD.integer)
    add_literal(g, movie_uri, BDA.imdbId, row['imdb_id'], XSD.string)
    
    # Link to collection
    if pd.notna(row['collection_id']):
        collection_uri = INST[f'collection/{row["collection_id"]}']
        g.add((collection_uri, RDF.type, BDA.MovieCollection))
        g.add((collection_uri, SCHEMA.name, Literal(row['collection_name'], datatype=XSD.string)))
        g.add((movie_uri, BDA.belongsToCollection, collection_uri))

print("Populating from 'movie_genres'...")
for _, row in movie_genres_df.iterrows():
    movie_uri = INST[f'movie/{row["movie_id"]}']
    genre_uri = INST[f'genre/{row["genre_id"]}']
    g.add((genre_uri, RDF.type, BDA.Genre))
    add_literal(g, genre_uri, SCHEMA.name, row['genre_name'], XSD.string)
    g.add((movie_uri, BDA.hasGenre, genre_uri))

print("Populating from 'movie_cast'...")
for _, row in movie_cast_df.iterrows():
    movie_uri = INST[f'movie/{row["movie_id"]}']
    actor_uri = INST[f'person/{row["actor_id"]}']
    g.add((actor_uri, RDF.type, BDA.Actor))
    add_literal(g, actor_uri, FOAF.name, row['actor_name'], XSD.string)
    add_literal(g, actor_uri, FOAF.gender, row['gender'], XSD.integer)
    g.add((movie_uri, BDA.hasActor, actor_uri))
    
print("Populating from 'movie_crew'...")
for _, row in movie_crew_df.iterrows():
    movie_uri = INST[f'movie/{row["movie_id"]}']
    crew_uri = INST[f'person/{row["crew_member_id"]}']
    g.add((crew_uri, RDF.type, BDA.CrewMember))
    add_literal(g, crew_uri, FOAF.name, row['crew_member_name'], XSD.string)
    add_literal(g, crew_uri, FOAF.gender, row['gender'], XSD.integer)
    add_literal(g, crew_uri, BDA.job, row['crew_job'], XSD.string)
    add_literal(g, crew_uri, BDA.department, row['department'], XSD.string)
    g.add((movie_uri, BDA.hasCrewMember, crew_uri))

print("Integrating data from 'omdb_details'...")
imdb_to_uri = {row['imdb_id']: INST[f'movie/{row["movie_id"]}'] for _, row in movies_df.dropna(subset=['imdb_id']).iterrows()}
for _, row in omdb_details_df.iterrows():
    if row['imdb_id'] in imdb_to_uri:
        movie_uri = imdb_to_uri[row['imdb_id']]
        add_literal(g, movie_uri, SCHEMA.boxOffice, row['box_office'], XSD.string)
        add_literal(g, movie_uri, BDA.metascore, row['metascore'], XSD.integer)
        
        if pd.notna(row['awards']):
            awards_text = row['awards'].lower()
            wins = re.search(r'(\d+) wins', awards_text)
            nominations = re.search(r'(\d+) nominations', awards_text)
            if wins:
                g.add((movie_uri, BDA.awardWins, Literal(int(wins.group(1)), datatype=XSD.integer)))
            if nominations:
                g.add((movie_uri, BDA.awardNominations, Literal(int(nominations.group(1)), datatype=XSD.integer)))

print("Populating from 'actor_articles'...")
for _, row in actor_articles_df.iterrows():
    actor_uri = INST[f'person/{row["actor_id"]}']
    article_uri = INST[f'article/{quote_plus(row["web_url"]) }']
    g.add((article_uri, RDF.type, BDA.Article))
    add_literal(g, article_uri, SCHEMA.name, row['title'], XSD.string)
    add_literal(g, article_uri, SCHEMA.description, row['abstract'], XSD.string)
    add_literal(g, article_uri, SCHEMA.datePublished, row['pub_date'], XSD.date)
    g.add((actor_uri, BDA.isDiscussedIn, article_uri))

print("Populating from 'music_tracks'...")
for _, row in music_tracks_df.iterrows():
    movie_uri = INST[f'movie/{row["movie_id"]}']
    album_uri = INST[f'album/{quote_plus(row["album_name"])}_{quote_plus(row["artist_name"]) }']
    track_id = row["track_id"].replace(' ','').replace('.','').replace('"','')
    track_uri = INST[f'track/{track_id}']
    artist_id = row["artist_id"].replace(' ','')
    artist_uri = INST[f'artist/{artist_id}']
    
    # Album
    g.add((album_uri, RDF.type, BDA.SoundtrackAlbum))
    g.add((movie_uri, BDA.hasSoundtrackAlbum, album_uri))
    add_literal(g, album_uri, SCHEMA.name, row['album_name'], XSD.string)
    
    # Track
    g.add((track_uri, RDF.type, BDA.SoundtrackTrack))
    add_literal(g, track_uri, SCHEMA.name, row['track_name'], XSD.string)
    add_literal(g, track_uri, SCHEMA.duration, row['track_duration_ms'], XSD.integer)
    g.add((album_uri, BDA.hasTrack, track_uri))

    # Artist
    g.add((artist_uri, RDF.type, BDA.Artist))
    add_literal(g, artist_uri, FOAF.name, row['artist_name'], XSD.string)
    g.add((track_uri, BDA.performedBy, artist_uri))

print("\nGraph population complete.")

Populating graph from 'movies' table...
Populating from 'movie_genres'...
Populating from 'movie_cast'...
Populating from 'movie_crew'...
Integrating data from 'omdb_details'...
Populating from 'actor_articles'...
Populating from 'music_tracks'...

Graph population complete.


## 4. Verification and Serialization

Let's run a quick SPARQL query to verify that our graph has been built correctly. We will find all actors for a specific, well-known movie. Then, we serialize the entire graph to a file.

In [100]:
print("--- Verifying graph with a SPARQL query ---")

query = """
    PREFIX bda: <http://www.bdaproject.org/schema#>
    PREFIX foaf: <http://xmlns.com/foaf/0.1/>
    PREFIX schema: <http://schema.org/>

    SELECT *
    WHERE {
        <http://www.bdaproject.org/instance/movie/120> schema:name ?name .
    }
    LIMIT 5
"""

results = g.query(query)
print("Output:")
for row in results:
    print(row.name)


--- Verifying graph with a SPARQL query ---
Output:
The Lord of the Rings: The Fellowship of the Ring


In [101]:
print("\n--- Querying for Star Wars soundtrack ---")


soundtrack_query = """
    PREFIX bda: <http://www.bdaproject.org/schema#>
    PREFIX schema: <http://schema.org/>

    SELECT ?trackName
    WHERE {
      ?movie schema:name ?moviename .
      ?movie bda:hasSoundtrackAlbum ?album .
      ?album bda:hasTrack ?track .
      ?track schema:name ?trackName .
    FILTER regex(?moviename, "Star Wars.*", "i")
    }
   
    ORDER BY ?trackName
"""

soundtrack_results = g.query(soundtrack_query)

print("Soundtrack for movie:")
track_count = 0
for row in soundtrack_results:
    print(f"- {row.trackName}")
    track_count += 1

if track_count == 0:
    print("No soundtrack found for movie. Check if the movie and its music data are in the graph.")


--- Querying for Star Wars soundtrack ---
Soundtrack for movie:
- A New Hope and End Credits
- Anakin vs. Obi-Wan
- Anakin's Betrayal
- Anakin's Dark Deeds
- Anakin's Dream
- Battle of the Heroes
- Enter Lord Vader
- General Grievous
- Grievous Speaks to Lord Sidious
- Grievous and the Droids
- Padmé's Ruminations
- Palpatine's Teachings
- Star Wars and the Revenge of the Sith
- The Birth of the Twins and Padmé's Destiny
- The Immolation Scene


In [102]:
output_path = "knowledge_graph.ttl"
print(f"\nSerializing graph with {len(g)} triples to {output_path}...")
g.serialize(destination=output_path, format='turtle')
print("Serialization complete. The KG is ready to be loaded into a triple store.")


Serializing graph with 42302 triples to knowledge_graph.ttl...
Serialization complete. The KG is ready to be loaded into a triple store.
